In [1]:
# TODO make sure this renders in the github repo

# Configurations For The Llama 3 Models

![chart](./showcase_images/model_sizes_chart.png)


- **Layers:** Noted as **num_layers**. How many Decoder layers the model has.
- **Model Dimensions:** Also noted as **d_model** or $d_{model}$. It represents the expected features for each token in the input and output sequences.
    - Example: when the Token Embeddings layer happen, each token is converted into a dense vector of size $d_{model}$.
- **FFN Dimensions:** Noted as **d_ff**. Is the hidden size of the Feed Forward SwiGLU sub-layer. We use this to expand the linear layers for SwiGLU, which gives the model more parameters to learn non-linear functions.
- **Attention Heads:** This the the total number of Query heads, note K & V use the Key/Value Heads hyperparameter.
    - If d_model = $4{,}096$ and attn_heads = $32$, each individual head processes a vector of size $4{,}096/32 = 128$
- **Key/Value Heads:** Noted as **num_kv_heads**. The number of key and value heads in the attention mechanism. The reason the number of Key and Value heads is smaller than the number of Query heads, is because this allows faster generation, larger batch sizes, and reduces the memory footprint.
- **Peak Learning Rate:** At the end of the warmup, the learning rate hits the peak it can be, then it starts to decay. This keeps training stable, and prevents exploding gradients early on when the weights are random.

- **Vocabulary Size:** The model's dictionary. It represents the words/tokens model can understand. It is static.
    - **Context Window length/sequence_len:** The model's short-term memory. Represents how much words/tokens the model can remember at one time. So, as you chat with the model, it loses track of earlier context that no longer fits its context window. In the data, it is the number of tokens in a single document/row in a list of documents.

- **Positional Embeddings** The frequency that the Q and K tokens are rotated by, to give them positional information.


The [Llama-3.1 8B tokenizer](https://huggingface.co/meta-llama/Llama-3.1-8B/blob/main/tokenizer_config.json) consists of $128{,}255$ tokens.

- **Standard tokens:** IDs $0$ through $127{,}999$ ($128{,}000$ total) are the standard tokens.
- **Special tokens:** IDs $128{,}000$ through $128{,}255$ ($256$ total) are the special tokens.
- **Llama 3 paper:** "We use a vocabulary with 128K tokens. Our token vocabulary combines 100K tokens from the tiktoken tokenizer with 28K additional tokens to better support non-English languages. Compared to the Llama 2 tokenizer, our new tokenizer improves compression rates on a sample of English data from 3.17 to 3.94 characters per token. This enables the model to “read” more text for the same amount of training compute. We also found that adding 28K tokens from select non-English languages improved both compression ratios and downstream performance, with no impact on English tokenization."


In [2]:
# TODO At end of project, make sure tree below is correct!

**Tree structure:**

```text
.
├── artifacts
│   ├── checkpoints
│   │   ├── Scaled_down_Base
│   │   │   ├── global_step_000250
│   │   │   │   ├── config.json
│   │   │   │   ├── training_loss_plot.png
│   │   │   │   └── checkpoint.pt
│   │   │   └── global_step_000010
│   │   │       ├── config.json
│   │   │       ├── training_loss_plot.png
│   │   │       └── checkpoint.pt
│   │   ├── Scaled_down_SFT
│   │   │   └── *
│   │   │   
│   │   └── Scaled_down_Instruct (Inference models)
│   │       └── *
│   │       
│   ├── meta-models # Meta's trained models # TODO will I save them locally or just load in memory?
│   ├── models # My trained models as Safetensors
│   └── universal_tokenizer # For my scaled down model
│       ├── bpe_vocab_size_32000_samples_1000_info.json
│       └── bpe_vocab_size_32000_samples_1000.json
└── data
    ├── initial_and_lc_stage.bin
    ├── initial_and_lc_stage.json
    ├── initial_and_lc_stage.parquet
    *
```


# Configurations:


In [ ]:
import easyjupyter

from pathlib import Path
import torch
from utils.fs_manager import find_root, create_folder_structure
from utils.misc import  print_config, get_device, get_num_workers
from pydantic import BaseModel, Field, PrivateAttr, computed_field
from typing import Optional, ClassVar, Literal
import numpy as np
import math

## Configurations Template

In [ ]:
class DatasetConfig(BaseModel):
    name: str
    config: Optional[str] = None


class StageTokensConfig(BaseModel):
    """Holds fields that are used for all pre-training stages, makes the configs code dryer"""

    tokens: int = Field(
        description="The number of tokens that a stage needs, so that during a training run, the model can be train on set number of tokens."
    )
    val_tokens: Optional[int] = Field(
        description="The number of tokens that a stage needs, to validated. Note: this is for the validation that occurs in the training loop."
    )

    tokens_processed: int = Field(
        default=0,
        description="Track the number of tokens that a model has been trained on so far. So, that when I continue training from a checkpoint, I don't feed the model the same data again.",
    )

    steps_completed: int = Field(
        default=0,
        description="Steps completed within the active stage. Needed for the initial stage learning rate schedule",
    )

    streaming_batch_size: int = Field(
        description="The batch size for streaming from HuggingFace. Datasets that contain longer sequences needs to stream with a small batch size. For example, the PG19 contains long books, so its streaming_batch_size should be smaller, while the FineWeb contains smaller sequences, its streaming_batch_size should be bigger."
    )

    def is_complete(self) -> bool:
        """Returns True if the stage has hit its target tokens or steps."""
        raise NotImplementedError("Subclasses must implement is_complete()")

    def get_target_shapes(
        self, current_seq: int, current_global_tokens: int
    ) -> tuple[int, int, int]:
        """Returns the (seq_len, global_batch_size_tokens) expected at the current progress."""
        raise NotImplementedError("Subclasses must implement get_target_shapes()")

    def get_learning_rate(self, initial_min_lr) -> float:
        """Returns the specific absolute learning rate for this stage depending on the step or tokens consumed.

        Args:
            initial_min_lr: The lowest the learning got to during the decay lr of the initial stage.
        """
        raise NotImplementedError("Subclasses must implement get_learning_rate()")


class InitialStageConfig(StageTokensConfig):
    """Initial stage of the text-only pre-training."""

    def get_target_shapes(
        self, current_seq: int, current_global_tokens: int
    ) -> tuple[int, int, int]:
        """
        The batch size and sequence length is doubled after a certain number of steps in the initial stage.
             Paper: "Specifically, we use an initial batch size of 4M tokens and sequences of length 4,096, and double these values to a batch size of 8M sequences of 8,192 tokens after pre-training 252M tokens. We double the batch size again to 16M after pre-training on 2.87T tokens."

        Args:
            current_seq: The current sequence length.
            current_global_tokens: The current global batch size in tokens, e.g., initially 4M for the 405B model.
        Returns:
            (seq_len, current_global_tokens)
        """
        if self.tokens_processed < self.first_increase_at:
            # Paper: "we use an initial batch size of 4M tokens and sequences of length 4,096"
            return self.initial_seq_len, self.initial_global_batch_size_tokens
        elif self.tokens_processed < self.second_increase_at:
            # Paper: "double these values to a batch size of 8M sequences of 8,192 tokens after pre-training 252M tokens"
            # print( f"Doubling batch_size and seq_len to {self.initial_batch_size * 2, self.initial_seq_len * 2}")
            return self.initial_seq_len * 2, self.initial_global_batch_size_tokens * 2

        else:
            # Paper: "We double the batch size again to 16M after pre-training on 2.87T tokens."
            # print(f"Doubling batch_size to {self.initial_batch_size * 4}")
            return self.initial_seq_len * 2, self.initial_global_batch_size_tokens * 4

    def is_complete(self) -> bool:
        return self.steps_completed >= self.steps

    def get_learning_rate(self, initial_min_lr: float) -> float:
        # 1. Linear warmup to peak learning rate
        if self.steps_completed < self.warmup_steps:
            return self.peak_lr * (
                float(self.steps_completed) / float(max(1, self.warmup_steps))
            )

        # 2. Cosine decay over remain steps
        decay_progress = float(self.steps_completed - self.warmup_steps) / float(
            max(1, self.steps - self.warmup_steps)
        )
        decay_progress = min(1.0, decay_progress)
        cosine_decay = 0.5 * (1.0 + math.cos(math.pi * decay_progress))
        return self.decay_lr_to + (self.peak_lr - self.decay_lr_to) * cosine_decay

    steps: int = Field(
        description="The initial stage is trained over 1.2M steps for the 405B model."
    )
    warmup_steps: int

    peak_lr: float
    decay_lr_to: float

    initial_seq_len: int
    initial_global_batch_size_tokens: int = Field(
        description="The target tokens that are consumed per optimizer step. For example, the 405B model has an initial batch size of 4M, we can not fit 4M into memory, so we use a micro batch size to accumulate gradients in a loop, until we reach the target tokens (40M), and then finally do a single optimizer step."
    )

    first_increase_at: int = Field(
        description="After training on the first 252M tokens, the batch_size and seq_len are doubled to 8M and 8,192 respectively."
    )
    second_increase_at: int = Field(
        description="After training on 2.87T tokens, the batch_size is doubled again to 16M"
    )

    dataset: DatasetConfig = Field(
        default_factory=lambda: DatasetConfig(
            name="HuggingFaceFW/fineweb-edu",
            config="sample-10BT",
        )
    )

    tokenizer_training_samples: int = Field(
        description="How many samples/documents to stream from HuggingFace dataset to train a tokenizer with."
    )


class Lc_StageConfig(StageTokensConfig):
    """Long-Context stage of the text-only pre-training."""

    seq_len: int
    increase_seq_len_to: int = Field(
        description="Paper: 'we increased context length gradually in six stages, starting from the original 8K context window and ending in the final 128K context window'. NOTE: context length == seq_len"
    )

    dataset: DatasetConfig = Field(
        description="The long-context stage requires a data mix that contains long sequences.",
        default_factory=lambda: DatasetConfig(
            name="emozilla/pg19",
        ),
    )

    @computed_field
    @property
    def seq_len_milestones(self) -> list[int]:
        """Calculates the milestones where we increased context length (seq_len) gradually in six stages of the long-context"""
        n = 6
        start = self.seq_len
        end = self.increase_seq_len_to
        return [
            # if 405B config from 8,192 to 128K, stages = [8192, 32154, 56115, 80077, 104038, 128000]
            round(start + (end - start) * i / (n - 1))
            for i in range(n)
        ]

    def is_complete(self) -> bool:
        return self.tokens_processed >= self.tokens

    def get_target_shapes(
        self, current_seq: int, current_global_tokens: int
    ) -> tuple[int, int]:
        """Paper: 'we increased context length gradually in six stages, starting from the original 8K context window and ending in the final 128K context window'. NOTE: context length == seq_len"""

        # Calculate how many tokens each milestone lasts
        tokens_per_milestones = self.tokens / len(self.seq_len_milestones)

        # Find which window (inside the [8,192 ••• 128K] divided into 6 parts) the current token count lands in
        milestone_idx = min(
            int(self.tokens_processed // tokens_per_milestones),
            len(self.seq_len_milestones) - 1,
        )

        # Inherit the global_tokens from whatever the initial stage ended at. Only update seq_len
        return (
            self.seq_len_milestones[milestone_idx],
            current_global_tokens,
        )

    def get_learning_rate(self, initial_min_lr: float) -> float:
        return initial_min_lr  # Learning rate isn't changed during this stage


class AnnealingStageConfig(StageTokensConfig):
    """Annealing stage of the text-only pre-training."""

    decay_lr_to: float
    seq_len: int
    dataset: DatasetConfig = Field(
        default_factory=lambda: DatasetConfig(
            name="HuggingFaceFW/finepdfs",
            config="eng_Latn",
        )
    )

    def is_complete(self) -> bool:
        return self.tokens_processed >= self.tokens

    def get_target_shapes(
        self, current_seq: int, current_global_tokens: int
    ) -> tuple[int, int]:
        # Frozen shapes. Inherits entirely from the end of the long-context stage, as the annealing does not change any shapes.
        return current_seq, current_global_tokens

    def get_learning_rate(self, initial_min_lr: float) -> float:
        """initial_min_lr: The minimum lr that was set in the initial stage."""
        # Linear decay to 0 over the token of this stage
        decay_progress = float(self.tokens_processed) / float(max(1, self.tokens))
        decay_progress = min(1.0, decay_progress)

        # Linearly decay from min_lr down to target 0.0
        target_lr = self.decay_lr_to
        return initial_min_lr - ((initial_min_lr - target_lr) * decay_progress)


class PretrainConfig(BaseModel):
    initial_stage: InitialStageConfig
    lc_stage: Lc_StageConfig
    annealing_stage: AnnealingStageConfig

    curr_stage_name: Literal[
        "initial_stage", "lc_stage", "annealing_stage", "complete"
    ] = Field(
        default="initial_stage",
        description="The active stage. Saved to JSON to allow seamless resumes.",
    )
    save_dir_name: str = Field(description="Example: Scaled_down_Base")

    transition_steps: dict[str, int] = Field(
        default_factory=dict,
        description="Track the step where we transitioned into the next stage, this is solely for the plotting accurate stage vertical lines on the training_loss_plot.png.",
    )


class TextOnlyConfig(BaseModel):
    pretrain: PretrainConfig
    train_sft: Optional[dict] = None  # TODO
    train_dpo: Optional[dict] = None  # TODO


class ConfigTemplate(BaseModel):

    _initialized: bool = PrivateAttr(default=False)

    # 🚨 NOTE: For NVIDIA GPUs the Golden Rule is: num_worker = 4 * num_GPU | On Mac Silicone even though I have a 32 core GPU, it is still only one GPU, best to set num_workers = 0.
    num_workers: int = Field(default_factory=get_num_workers)

    model_name: str
    rope_theta: int
    rms_norm_eps: float
    d_model: int
    num_layers: int
    d_ff: int
    attn_heads: int
    num_kv_heads: int
    vocab_size: int

    is_overfit:bool = Field(default=False, description="Whether this config is for an overfit test.")
    prefix: str = Field(
        default="",
        description="Prefix to append to a new directory, e..g, prefix='Overfit_' -> Overfit_<model_name>",
    )

    dataset_dtype_str: Literal["uint16", "uint32"] = Field(
        default="uint16",
        description="Data type for storing the binary dataset. If custom BPE tokenizer vocab is < 65K, use np.uint16 to save 50% of space. The Llama3 vocab was 128K so it must use np.uint32.",
    )

    num_grad_updates: int = Field(
        default=0,
        description="The total number of gradient updates across all stages. Used solely naming checkpoints and logging metrics. The main reason I use `num_grad_updates` is because some of the phases use different methods; for example, the initial stage is trained on a number of steps, but the LC and annealing stages are trained for a number of tokens.",
    )

    text_only: TextOnlyConfig
    multimodal: Optional[dict] = None  # TODO

    # The `seq_len` and `batch_size` sizes are altered during training
    curr_seq_len: int = Field(default=0)
    curr_global_batch_tokens: int = Field(
        default=0,
        description="Global batch size in tokens that the model needs to see before we can do a single optimizer step.",
    )
    curr_micro_batch_size: int = Field(
        default=0,
        description="The maximum number of sequences that can fit into VRAM during a single forward pass. Used for gradient accumulation, we accumulate gradients until we reach the size of the curr_global_batch_tokens, then we do a single optimizer step.",
        exclude=True,
    )

    # Managing fs resources, excluded form serialization
    _project_root: Path = PrivateAttr(default_factory=find_root)
    _data_dir: Path = PrivateAttr()
    _artifacts_dir: Path = PrivateAttr()
    _chpts_dir: Path = PrivateAttr()
    _device: torch.device = PrivateAttr(default_factory=get_device)

    special_tokens: dict[str, dict[str, str | int]] = Field(
        default_factory=lambda: {
            "pad_token": {"token": "<|pad_token|>", "ID": 0},
            # The token for the beginning of the text.
            "doc_begin_token": {"token": "<|begin_of_doc|>", "ID": 1},
            # The token for the end of document, it is injected between documents.
            "doc_end_token": {"token": "<|end_of_doc|>", "ID": 2},
            # unknown token.
            "unk_token": {"token": "<|unk|>", "ID": 3},
        }
    )

    def get_chpt_save_path(self, chpt_dir_name: str, optionalPrefix="") -> Path:
        chpt_dir_name = f"{self.prefix}{chpt_dir_name}"
        save_path = self.chpts_dir / chpt_dir_name / f"{optionalPrefix}num_grad_updates_{self.num_grad_updates}"
        save_path.mkdir(parents=True, exist_ok=True)
        return save_path

    def initialize(self, is_overfit=False):
        """
        Skip initialization at definition, this way I can implement a model like the `Llama3_1_405B = ConfigTemplate` in this file and then in main.py, initialize it with `.initialize()` to explicitly initialize the config in main.py. Otherwise, it would initialize the config just by doing `from configs import Llama3_1_405B`
        """
        self._data_dir, self._artifacts_dir, self._chpts_dir = create_folder_structure(
            self._project_root
        )
        print(f"Using config: {self.model_name}")

        if is_overfit:
            print("Configured for overfitting...")
            self.prefix = "Overfit_"
            self.is_overfit=True

            # --- Lower parameters count
            self.d_model = 128
            self.num_layers = 2
            self.attn_heads = 4
            self.num_kv_heads = 4
            self.d_ff = 512

            # --- initial
            self.text_only.pretrain.initial_stage.steps = 100
            self.text_only.pretrain.initial_stage.warmup_steps = 10
            self.text_only.pretrain.initial_stage.initial_global_batch_size_tokens = 256
            self.text_only.pretrain.initial_stage.initial_seq_len = 32

            self.text_only.pretrain.initial_stage.first_increase_at = 5_000
            self.text_only.pretrain.initial_stage.second_increase_at = 8_000

            self.text_only.pretrain.initial_stage.tokens = 20_000
            self.text_only.pretrain.initial_stage.val_tokens = 5_000


            self.text_only.pretrain.initial_stage.peak_lr = 1e-3
            # bring the decay_lr_to from 0 to a higher number for overfit
            self.text_only.pretrain.initial_stage.decay_lr_to = 1e-4

            self.text_only.pretrain.initial_stage.tokenizer_training_samples = 100
            self.text_only.pretrain.initial_stage.streaming_batch_size = 100

            # --- LC stage
            self.text_only.pretrain.lc_stage.seq_len = 64
            self.text_only.pretrain.lc_stage.increase_seq_len_to = 256

            self.text_only.pretrain.lc_stage.tokens = 10_000
            self.text_only.pretrain.lc_stage.val_tokens = 2_000
            self.text_only.pretrain.lc_stage.streaming_batch_size = 1

            # --- Annealing
            self.text_only.pretrain.annealing_stage.tokens = 7_000
            self.text_only.pretrain.annealing_stage.val_tokens = 1_000
            self.text_only.pretrain.annealing_stage.seq_len = 256
            self.text_only.pretrain.annealing_stage.streaming_batch_size = 50

        return self

    @property
    def project_root(self) -> Path:
        return self._project_root

    @property
    def data_dir(self) -> Path:
        return self._data_dir

    @property
    def artifacts_dir(self) -> Path:
        return self._artifacts_dir

    @property
    def chpts_dir(self) -> Path:
        return self._chpts_dir

    @property
    def device(self) -> torch.device:
        return self._device

    def dataset_dtype(self):
        return np.uint16 if self.dataset_dtype_str == "uint16" else np.uint32

    def save(self, save_path: Path):
        """Native Pydantic serialization"""
        save_path.parent.mkdir(parents=True, exist_ok=True)
        with open(save_path / "config.json", "w") as f:
            f.write(self.model_dump_json(indent=4))
        print(f"Saved Config to -> {save_path / "config.json"}")

    @classmethod
    def load(cls, chpt_dir: Path):
        """Native Pydantic deserialization"""
        with open(chpt_dir / "config.json", "r") as f:
            instance = cls.model_validate_json(f.read())
        print(f"Loaded Config from -> {chpt_dir / "config.json"}")
        return instance

    def __str__(self) -> str:
        return print_config(self)

    def __repr__(self) -> str:
        return print_config(self)

## Llama 3.1 405B Configuration

💡 The 405B model is the largest model from the Llama 3 family, with 405 Billion parameters, it is a text-only model, and is also the most well documented in the Llama 3 paper. I will not be training this model, as it is too large. Instead, I will create the configuration for the 405B model, build out the training pipeline, and then create a scaled down configuration version of it, and train that instead.

**Llama 3 paper:** "**Language model pre-training.** We start by converting a large, multilingual text corpus to discrete tokens and pre-training a large language model (LLM) on the resulting data to perform next-token prediction. In the language model pre-training stage, the model learns the structure of language and obtains large amounts of knowledge about the world from the text it is “reading”. To do this effectively, pre-training is performed at massive scale: we pre-train **a model with 405B parameters** on **15.6T tokens using a context window of 8K tokens**. This standard pre-training stage is followed by a continued pre-training stage that increases the supported **context window to 128K tokens**."


---

**PAPER:**

**3.4.1 Initial Pre-Training**

We pre-train Llama $3$ $405\text{B}$ using AdamW with a peak learning rate of $8 × 10^{−5}$ , a linear warm up of $8{,}000$ steps, and a cosine learning rate schedule decaying to $8 × 10^{−7}$ over $1{,}200{,}000$ steps. We use a lower batch size early in training to improve training stability, and increase it subsequently to improve efficiency. Specifically, we use an initial batch size of $4\text{M}$ tokens and sequences of length $4{,}096$, and double these values to a batch size of $8\text{M}$ sequences of $8{,}192$ tokens after pre-training $252\text{M}$ tokens. We double the batch size again to 16M after pre-training on $2.87\text{T}$ tokens. We found this training recipe to be very stable: we observed few loss spikes and did not require interventions to correct for model training divergence.

**3.4.2 Long Context Pre-Training**

In the final stages of pre-training, we train on long sequences to support context windows of up to $128\text{K}$ tokens. We do not train on long sequences earlier because the compute in self-attention layers grows quadratically in the sequence length. We increase the supported context length in increments, pre-training until the model has successfully adapted to the increased context length. We assess successful adaptation by measuring whether (1) model performance on short-context evaluations has recovered completely and (2) the model perfectly solves “needle in a haystack” tasks up to that length. In Llama $3$ $405\text{B}$ pre-training, we increased context length gradually in six stages, starting from the original $8\text{K}$ context window and ending in the final $128\text{K}$ context window. This long-context pre-training stage was performed using approximately $800\text{B}$ training tokens.

**3.4.3 Annealing**

During pre-training on the final $40\text{M}$ tokens, we linearly annealed the learning rate to $0$, maintaining a context length of $128\text{K}$ tokens. During this annealing phase, we also adjusted the data mix to upsample data sources of very high quality; see Section 3.1.3. Finally, we compute the average of model checkpoints (Polyak (1991) averaging) during annealing to produce the final pre-trained model.

---

🚨 **Notes:**

- For attributes like the value for the RMSNorm `rms_norm_eps`, which are not mentioned in the paper, I referenced from the official [HuggingFace Meta 405B config.json](https://huggingface.co/meta-llama/Llama-3.1-405B/blob/main/config.json)

In [5]:
Llama3_1_405B = ConfigTemplate(
    # Llama 3.1 405B Configuration. This model is not multi-modal, it only takes in text!
    model_name="Llama 3.1 405B",
    rope_theta=500_000,
    rms_norm_eps=1e-5,
    d_model=16_384,
    num_layers=126,
    dataset_dtype_str="uint32",
    d_ff=53_248,
    attn_heads=128,
    num_kv_heads=8,
    vocab_size=128_000,
    text_only=TextOnlyConfig(
        pretrain=PretrainConfig(
            save_dir_name="405B_Base",
            initial_stage=InitialStageConfig(
                steps=1_200_000,
                warmup_steps=8_000,
                tokens=15_600_000_000_000,
                val_tokens=None,
                tokenizer_training_samples=50_000,
                initial_global_batch_size_tokens=4_000_000,
                initial_seq_len=4_096,
                first_increase_at=252_000_000,
                second_increase_at=2_870_000_000_000,
                peak_lr=8e-5,
                decay_lr_to=8e-7,
                streaming_batch_size=1_000,
            ),
            lc_stage=Lc_StageConfig(
                tokens=800_000_000_000,
                val_tokens=None,
                seq_len=8_192,
                increase_seq_len_to=128_000,
                streaming_batch_size=10,
            ),
            annealing_stage=AnnealingStageConfig(
                seq_len=128_000,
                tokens=40_000_000,
                val_tokens=None,
                decay_lr_to=0.0,
                streaming_batch_size=500,
            ),
        )
    ),
)

In [6]:
# @i-c
Llama3_1_405B.initialize()
print(Llama3_1_405B)


Project Root: /Users/tonyavis/Main/Build_an_LLM
Using config: Llama 3.1 405B


==================== Config ====================
            [Model Architecture]
                - model_name               = Llama 3.1 405B
                - d_model                  = 16384
                - num_kv_heads             = 8
                - d_ff                     = 53248
                - num_layers               = 126
                - attn_heads               = 128
                - rms_norm_eps             = 1e-05
                - rope_theta               = 500000
                - vocab_size               = 128000
            

            [Active Run]
                - num_workers              = 0
                - device                   = mps
            

            [Stages Configs]
                [Pre-Training Stage]
                    - token_budget         = None
                    - peak_lr              = None
                    


                    > Initial Stage:
 

## Scaled Down Llama 3.1 Architecture

- Scaled down version of the Llama 3.1 architecture.

In [7]:
Scaled_down_text = ConfigTemplate(
    # This model is not multi-modal, it only takes in text!
    model_name="scaled_down_text",
    rope_theta=500_000,
    rms_norm_eps=1e-5,
    d_model=2048,
    num_layers=16,
    d_ff=6_144, 
    dataset_dtype_str="uint16",
    attn_heads=16,
    num_kv_heads=4,
    vocab_size=64_000,
    text_only=TextOnlyConfig(
        pretrain=PretrainConfig(
            save_dir_name="Scaled_down_Base",
            initial_stage=InitialStageConfig(
                steps=11_522,
                warmup_steps=500,
                tokens=9_500_000_000,
                val_tokens=10_240_000,
                tokenizer_training_samples=50_000,
                initial_global_batch_size_tokens=250_000,
                initial_seq_len=2_048,
                first_increase_at=161_000_000,
                second_increase_at=1_700_000_000,
                peak_lr=1e-4,
                decay_lr_to=8e-7,
                streaming_batch_size=500,
            ),
            lc_stage=Lc_StageConfig(
                tokens=490_000_000,
                val_tokens=2_048_000,
                seq_len=4_096,
                increase_seq_len_to=8_192,
                streaming_batch_size=5,
            ),
            annealing_stage=AnnealingStageConfig(
                seq_len=8_192,
                tokens=24_000_000,
                val_tokens=1_024_000,
                decay_lr_to=0.0,
                streaming_batch_size=100,
            ),
        )
    ),
)

In [8]:
# @i-c
print(Scaled_down_text.initialize())


Project Root: /Users/tonyavis/Main/Build_an_LLM
Using config: scaled_down_text


==================== Config ====================
            [Model Architecture]
                - model_name               = scaled_down_text
                - d_model                  = 2048
                - num_kv_heads             = 4
                - d_ff                     = 6144
                - num_layers               = 16
                - attn_heads               = 16
                - rms_norm_eps             = 1e-05
                - rope_theta               = 500000
                - vocab_size               = 64000
            

            [Active Run]
                - num_workers              = 0
                - device                   = mps
            

            [Stages Configs]
                [Pre-Training Stage]
                    - token_budget         = None
                    - peak_lr              = None
                    


                    > Initial Stage:
  

## Llama 3.2 11B Configuration.

This config is for **multi-modal for vision and text**!


In [9]:
#TODO maybe I can just use the same scaled down for the multi-model model